# Azure AI Language + Azure AI Search setup for Custom Question Answering

This notebook:
- creates or recovers an Azure AI Language resource (`TextAnalytics`)
- creates an Azure AI Search service
- fetches endpoints and keys
- writes the values into a local `.env` file for later notebooks

Update the config cell before running.

In [ ]:
from pathlib import Path
from datetime import datetime
import os, json, subprocess

In [ ]:
## ========= CONFIG =========
SUBSCRIPTION_ID = "00000000-0000-0000-0000-000000000000"
RESOURCE_GROUP = "sample-resource-group"
LOCATION = "eastus"
LANGUAGE_RESOURCE_NAME = "sample-resource-name"
SEARCH_SERVICE_NAME = "sample-resource-name"
LANGUAGE_SKU = "S"   # change to 'S' if your subscription/region rejects F0
SEARCH_SKU = "standard"  # for CQA, S1/Standard is the safer default
SEARCH_PARTITIONS = 1
SEARCH_REPLICAS = 1
ENV_FILE = Path('.env')
AZ_EXE = r"C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin\az.cmd"

In [ ]:
# ========= AZ HELPERS =========
if not os.path.exists(AZ_EXE):
    raise FileNotFoundError(f"Azure CLI executable not found at: {AZ_EXE}")

def az(args, expect_json=True):
    cmd = [AZ_EXE] + args + ["-o", "json" if expect_json else "tsv"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"Azure CLI command failed:\n"
            f"CMD: {' '.join(cmd)}\n"
            f"STDOUT:\n{result.stdout}\n"
            f"STDERR:\n{result.stderr}"
        )
    return json.loads(result.stdout) if expect_json else result.stdout.strip()

def az_try(args, expect_json=True):
    cmd = [AZ_EXE] + args + ["-o", "json" if expect_json else "tsv"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        return None
    return json.loads(result.stdout) if expect_json else result.stdout.strip()

print("Checking Azure CLI login...")
acct = az_try(["account", "show"])
if acct is None:
    raise RuntimeError(
        "Azure CLI is reachable, but not logged in.\n"
        "Run in terminal:\n"
        "  az login\n"
        "or:\n"
        "  az login --use-device-code"
    )

print("Logged in as:", acct.get("user", {}).get("name"))
print("Subscription:", acct.get("name"))

print("Setting Azure subscription...")
az(["account", "set", "--subscription", SUBSCRIPTION_ID], expect_json=False)

print(f"Ensuring resource group exists: {RESOURCE_GROUP}")
rg = az([
    "group", "create",
    "--name", RESOURCE_GROUP,
    "--location", LOCATION
])

print("Resource group ready:", rg["name"])

In [ ]:
# =========================
# .env UPSERT HELPER
# =========================
def upsert_env_file(path: Path, updates: dict):
    existing = {}

    if path.exists():
        for line in path.read_text(encoding='utf-8').splitlines():
            line = line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            k, v = line.split('=', 1)
            existing[k] = v.strip().strip('\"')

    existing.update({k: '' if v is None else str(v) for k, v in updates.items()})

    with path.open('w', encoding='utf-8') as f:
        for k, v in existing.items():
            f.write(f'{k}=\"{v}\"\n')


In [ ]:
# =========================
# ENSURE LANGUAGE RESOURCE
# =========================
def ensure_language_account(name, sku, location, resource_group):
    kind = 'TextAnalytics'

    existing = az_try([
        'cognitiveservices', 'account', 'show',
        '--name', name,
        '--resource-group', resource_group
    ])

    if existing is not None:
        print(f'Already exists: {name} ({kind})')
        return existing

    deleted = az_try([
        'cognitiveservices', 'account', 'show-deleted',
        '--name', name,
        '--resource-group', resource_group,
        '--location', location
    ])

    if deleted is not None:
        print(f'Recovering soft-deleted resource: {name} ({kind})')
        az([
            'cognitiveservices', 'account', 'recover',
            '--name', name,
            '--resource-group', resource_group,
            '--location', location
        ], expect_json=False)

        for _ in range(12):
            recovered = az_try([
                'cognitiveservices', 'account', 'show',
                '--name', name,
                '--resource-group', resource_group
            ])
            if recovered is not None:
                print(f'Recovered: {name} ({kind})')
                return recovered
            time.sleep(5)

        raise RuntimeError(f"Recovery started but '{name}' is not yet visible.")

    print(f'Creating: {name} ({kind})')
    return az([
        'cognitiveservices', 'account', 'create',
        '--name', name,
        '--resource-group', resource_group,
        '--kind', kind,
        '--sku', sku,
        '--location', location,
        '--yes'
    ])


In [ ]:
# =========================
# ENSURE SEARCH SERVICE
# =========================
def ensure_search_service(name, sku, location, resource_group, partition_count=1, replica_count=1):
    existing = az_try([
        'search', 'service', 'show',
        '--name', name,
        '--resource-group', resource_group
    ])

    if existing is not None:
        print(f'Already exists: {name} (Azure AI Search)')
        return existing

    print(f'Creating: {name} (Azure AI Search)')
    return az([
        'search', 'service', 'create',
        '--name', name,
        '--resource-group', resource_group,
        '--location', location,
        '--sku', sku,
        '--partition-count', str(partition_count),
        '--replica-count', str(replica_count)
    ])


In [ ]:
# =========================
# CREATE / ENSURE BOTH RESOURCES
# =========================
language_account = ensure_language_account(
    LANGUAGE_RESOURCE_NAME,
    LANGUAGE_SKU,
    LOCATION,
    RESOURCE_GROUP
)

search_service = ensure_search_service(
    SEARCH_SERVICE_NAME,
    SEARCH_SKU,
    LOCATION,
    RESOURCE_GROUP,
    partition_count=SEARCH_PARTITIONS,
    replica_count=SEARCH_REPLICAS
)

print('Azure AI Language and Azure AI Search resources ready.')


In [ ]:
# =========================
# FETCH LANGUAGE ENDPOINT + KEYS
# =========================
language_endpoint = az([
    'cognitiveservices', 'account', 'show',
    '--name', LANGUAGE_RESOURCE_NAME,
    '--resource-group', RESOURCE_GROUP,
    '--query', 'properties.endpoint'
], expect_json=False).strip().strip('\"')

language_keys = az([
    'cognitiveservices', 'account', 'keys', 'list',
    '--name', LANGUAGE_RESOURCE_NAME,
    '--resource-group', RESOURCE_GROUP
])

language_key1 = language_keys.get('key1')
language_key2 = language_keys.get('key2')

print('Language endpoint:', language_endpoint)
print('Language key1 fetched:', 'Yes' if language_key1 else 'No')
print('Language key2 fetched:', 'Yes' if language_key2 else 'No')


In [ ]:
# =========================
# FETCH SEARCH ENDPOINT + ADMIN KEYS
# =========================
search_endpoint = 'https://REDACTED_SEARCH_ENDPOINT/'

search_admin_keys = az([
    'search', 'admin-key', 'show',
    '--service-name', SEARCH_SERVICE_NAME,
    '--resource-group', RESOURCE_GROUP
])

search_primary_admin_key = search_admin_keys.get('primaryKey')
search_secondary_admin_key = search_admin_keys.get('secondaryKey')

print('Search endpoint:', search_endpoint)
print('Search primary admin key fetched:', 'Yes' if search_primary_admin_key else 'No')
print('Search secondary admin key fetched:', 'Yes' if search_secondary_admin_key else 'No')


In [ ]:
# =========================
# SAVE ALL VALUES TO .env
# =========================
upsert_env_file(ENV_FILE, {
    'AZURE_LANGUAGE_CQA_RESOURCE_NAME': LANGUAGE_RESOURCE_NAME,
    'AZURE_LANGUAGE_CQA_ENDPOINT': language_endpoint,
    'AZURE_LANGUAGE_CQA_KEY': language_key1,
    'AZURE_LANGUAGE_CQA_KEY2': language_key2,
    'AZURE_LANGUAGE_CQA_KIND': 'TextAnalytics',
    'AZURE_SEARCH_SERVICE_NAME': SEARCH_SERVICE_NAME,
    'AZURE_SEARCH_ENDPOINT': search_endpoint,
    'AZURE_SEARCH_ADMIN_KEY': search_primary_admin_key,
    'AZURE_SEARCH_ADMIN_KEY2': search_secondary_admin_key,
    'AZURE_SEARCH_SKU': SEARCH_SKU,
    'AZURE_RESOURCE_GROUP': RESOURCE_GROUP,
    'AZURE_LOCATION': LOCATION
})

print(f'Updated .env file at: {ENV_FILE.resolve()}')
